In [24]:
import requests  # Used to send API requests
import pandas as pd  # Used to organize data into tables
import matplotlib.pyplot as plt  # Used to create diagrams

## Thesis:
### In 2024, there is a negative relationship between population growth rate and GDP per capita across countries.

# Population grouwth data

In [25]:
url_page_1 = "https://api.worldbank.org/v2/country/all/indicator/SP.POP.GROW?format=json&per_page=500&page=1"

response_page_1 = requests.get(url_page_1)  # Send request for the first page
data_page_1 = response_page_1.json()  # Convert response into Python data

print(data_page_1[0])  # Show metadata

{'page': 1, 'pages': 36, 'per_page': 500, 'total': 17556, 'sourceid': '2', 'lastupdated': '2026-04-08'}


In [27]:
total_pages = data_page_1[0]['pages']  # Get the total number of pages
total_pages

36

In [28]:
all_records = []  # Create an empty list to store all records

for page in range(1, total_pages + 1):
    url = f"https://api.worldbank.org/v2/country/all/indicator/SP.POP.GROW?format=json&per_page=500&page={page}"
    response = requests.get(url)  # Send request for one page
    data = response.json()  # Convert response into Python data
    
    records = data[1]  # Get the actual records from this page
    all_records.extend(records)  # Add records to the full list
    
    print(f"Finished page {page}")  # Show progress

Finished page 1
Finished page 2
Finished page 3
Finished page 4
Finished page 5
Finished page 6
Finished page 7
Finished page 8
Finished page 9
Finished page 10
Finished page 11
Finished page 12
Finished page 13
Finished page 14
Finished page 15
Finished page 16
Finished page 17
Finished page 18
Finished page 19
Finished page 20
Finished page 21
Finished page 22
Finished page 23
Finished page 24
Finished page 25
Finished page 26
Finished page 27
Finished page 28
Finished page 29
Finished page 30
Finished page 31
Finished page 32
Finished page 33
Finished page 34
Finished page 35
Finished page 36


In [29]:
df_pop_raw = pd.DataFrame(all_records)  # Turn all records into a DataFrame
df_pop_raw.shape

(17556, 8)

In [31]:
df_pop = df_pop_raw[["country", "countryiso3code", "date", "value"]].copy()  # Keep only useful columns
df_pop.head()

,country,countryiso3code,date,value
0,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2025,NaN
1,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2024,2.472801
2,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2023,2.519167
3,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2022,2.592754
4,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2021,2.649439


In [33]:
country_names = []  # Create an empty list to store country names

for item in df_pop["country"]:
    country_names.append(item["value"])  # Get the name from each dictionary

df_pop["country_name"] = country_names  # Save the names as a new column

df_pop.head()  # Check the new column

,country,countryiso3code,date,value,country_name
0,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2025,NaN,Africa Eastern and Southern
1,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2024,2.472801,Africa Eastern and Southern
2,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2023,2.519167,Africa Eastern and Southern
3,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2022,2.592754,Africa Eastern and Southern
4,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2021,2.649439,Africa Eastern and Southern


In [35]:
df_pop["date"] = pd.to_numeric(df_pop["date"], errors="coerce")  # Convert year to numeric

print(df_pop["date"].dtype)  # Check the data type
df_pop["date"].head(10)

int64


0    2025
1    2024
2    2023
3    2022
4    2021
5    2020
6    2019
7    2018
8    2017
9    2016
Name: date, dtype: int64

In [36]:
df_pop_2024 = df_pop[df_pop["date"] == 2024].copy()  # Keep only rows for year 2024
df_pop_2024.shape

(266, 5)

In [37]:
df_pop_2024.isnull().sum()

country            0
countryiso3code    0
date               0
value              1
country_name       0
dtype: int64

In [38]:
df_pop_2024 = df_pop_2024.rename(columns={"value": "population_growth"})  # Rename the value column
df_pop_2024 = df_pop_2024.dropna(subset=["population_growth"])  # Remove missing values

df_pop_2024.shape

(265, 5)

In [39]:
df_pop_2024[["country_name", "countryiso3code", "date", "population_growth"]].head(20)

,country_name,countryiso3code,date,population_growth
1,Africa Eastern and Southern,AFE,2024,2.472801
67,Africa Western and Central,AFW,2024,2.398473
133,Arab World,ARB,2024,2.155917
199,Caribbean small states,CSS,2024,0.422919
265,Central Europe and the Baltics,CEB,2024,-0.121595
331,Early-demographic dividend,EAR,2024,1.151531
397,East Asia & Pacific,EAS,2024,0.162506
463,East Asia & Pacific (excluding high income),EAP,2024,0.168021
529,East Asia & Pacific (IDA & IBRD countries),TEA,2024,0.166308
595,Euro area,EMU,2024,0.399249


In [49]:
## download the country metadata and drop all the region data(keep only data from countries)

In [41]:
country_url = "https://api.worldbank.org/v2/country?format=json&per_page=400&page=1"

country_response = requests.get(country_url)
country_data = country_response.json() 

print(country_data[0])

{'page': 1, 'pages': 1, 'per_page': '400', 'total': 296}


In [42]:
country_records = country_data[1]  # Get the actual country metadata records

df_country_meta_raw = pd.DataFrame(country_records)  # Turn metadata into a DataFrame

df_country_meta_raw.head()

,id,iso2Code,name,region,adminregion,incomeLevel,lendingType,capitalCity,longitude,latitude
0,ABW,AW,Aruba,"{'id': 'LCN', 'iso2code': 'ZJ', 'value': 'Lati...","{'id': '', 'iso2code': '', 'value': ''}","{'id': 'HIC', 'iso2code': 'XD', 'value': 'High...","{'id': 'LNX', 'iso2code': 'XX', 'value': 'Not ...",Oranjestad,-70.0167,12.5167
1,AFE,ZH,Africa Eastern and Southern,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': ''}","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': 'Aggregates'}",,,
2,AFG,AF,Afghanistan,"{'id': 'MEA', 'iso2code': 'ZQ', 'value': 'Midd...","{'id': 'MNA', 'iso2code': 'XQ', 'value': 'Midd...","{'id': 'LIC', 'iso2code': 'XM', 'value': 'Low ...","{'id': 'IDX', 'iso2code': 'XI', 'value': 'IDA'}",Kabul,69.1761,34.5228
3,AFR,A9,Africa,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': ''}","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': 'Aggregates'}",,,
4,AFW,ZI,Africa Western and Central,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': ''}","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': 'Aggregates'}",,,


In [43]:
df_country_meta = df_country_meta_raw[["id", "name", "region", "incomeLevel"]].copy()  # Keep useful columns

region_names = []  # Create an empty list to store region names

for item in df_country_meta["region"]:
    region_names.append(item["value"])  # Get the region name from each dictionary

df_country_meta["region_name"] = region_names  # Save region names as a new column

df_country_meta.head()

,id,name,region,incomeLevel,region_name
0,ABW,Aruba,"{'id': 'LCN', 'iso2code': 'ZJ', 'value': 'Lati...","{'id': 'HIC', 'iso2code': 'XD', 'value': 'High...",Latin America & Caribbean
1,AFE,Africa Eastern and Southern,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...",Aggregates
2,AFG,Afghanistan,"{'id': 'MEA', 'iso2code': 'ZQ', 'value': 'Midd...","{'id': 'LIC', 'iso2code': 'XM', 'value': 'Low ...","Middle East, North Africa, Afghanistan & Pakistan"
3,AFR,Africa,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...",Aggregates
4,AFW,Africa Western and Central,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...",Aggregates


In [44]:
df_country_meta = df_country_meta.rename(columns={"id": "countryiso3code", "name": "meta_country_name"})  # Rename columns for merging

df_country_meta.head()

,countryiso3code,meta_country_name,region,incomeLevel,region_name
0,ABW,Aruba,"{'id': 'LCN', 'iso2code': 'ZJ', 'value': 'Lati...","{'id': 'HIC', 'iso2code': 'XD', 'value': 'High...",Latin America & Caribbean
1,AFE,Africa Eastern and Southern,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...",Aggregates
2,AFG,Afghanistan,"{'id': 'MEA', 'iso2code': 'ZQ', 'value': 'Midd...","{'id': 'LIC', 'iso2code': 'XM', 'value': 'Low ...","Middle East, North Africa, Afghanistan & Pakistan"
3,AFR,Africa,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...",Aggregates
4,AFW,Africa Western and Central,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...",Aggregates


In [45]:
df_country_only = df_country_meta[df_country_meta["region_name"] != "Aggregates"].copy()  # Keep only real countries

df_country_only.shape

(217, 5)

In [47]:
df_pop_2024_country = pd.merge(
    df_pop_2024,
    df_country_only[["countryiso3code", "region_name"]],
    on="countryiso3code",
    how="inner"
)  # Merge population data with real-country metadata

df_pop_2024_country.shape

(217, 6)

In [48]:
df_pop_2024_country[["country_name", "countryiso3code", "date", "population_growth", "region_name"]].head(20)

,country_name,countryiso3code,date,population_growth,region_name
0,Afghanistan,AFG,2024,2.836573,"Middle East, North Africa, Afghanistan & Pakistan"
1,Albania,ALB,2024,-1.543144,Europe & Central Asia
2,Algeria,DZA,2024,1.398387,"Middle East, North Africa, Afghanistan & Pakistan"
3,American Samoa,ASM,2024,-1.603666,East Asia & Pacific
4,Andorra,AND,2024,1.329307,Europe & Central Asia
5,Angola,AGO,2024,3.044200,Sub-Saharan Africa
6,Antigua and Barbuda,ATG,2024,0.487472,Latin America & Caribbean
7,Argentina,ARG,2024,0.345830,Latin America & Caribbean
8,Armenia,ARM,2024,2.307615,Europe & Central Asia
9,Aruba,ABW,2024,0.590657,Latin America & Caribbean


# GDP per capita data

In [50]:
gdp_url_page_1 = "https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD?format=json&per_page=500&page=1"

gdp_response_page_1 = requests.get(gdp_url_page_1)
gdp_data_page_1 = gdp_response_page_1.json()

print(gdp_data_page_1[0])

{'page': 1, 'pages': 36, 'per_page': 500, 'total': 17556, 'sourceid': '2', 'lastupdated': '2026-04-08'}


In [51]:
gdp_total_pages = gdp_data_page_1[0]["pages"]  # Get the total number of GDP pages
gdp_total_pages

36

In [ ]:
gdp_all_records = []  # Create an empty list to store all GDP records

for page in range(1, gdp_total_pages + 1):
    url = f"https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD?format=json&per_page=500&page={page}"
    response = requests.get(url)  # Send request for one GDP page
    data = response.json()  # Convert response into Python data
    
    records = data[1]  # Get the actual records from this page
    gdp_all_records.extend(records)  # Add records to the full list
    
    print(f"Finished GDP page {page}")  # Show progress